# Sketch-to-2D Studio — GPU Worker (Colab)

Notebook dựng 2 dịch vụ trên GPU và mở đường hầm `cloudflared` để máy Mac gọi tới:
- **ComfyUI** (vẽ nhân vật / icon, IP-Adapter khoá gương mặt) → `COMFY_URL`
- **Ollama + Qwen** (đọc ảnh, đọc chữ tiếng Việt, gợi ý — thay Claude Desktop) → `LLM_URL`

1. `Runtime → Change runtime type → GPU` (T4 miễn phí, hoặc L4/A100 nếu có Colab Pro).
2. Run all. Cell cuối in ra 2 link.
3. Trên Mac chạy `uv run studio-web`, mở http://127.0.0.1:8765 → **⚙︎ Cài đặt** → dán 2 link.
   (Vẫn dùng được Claude Desktop: chỉ cần COMFY_URL.)

Model tự chọn theo VRAM: **T4 → `qwen3.5:9b`** (6,6 GB), **≥ 40 GB (A100) → `qwen3.8:27b`** (18 GB). Qwen3.8 nhỏ nhất là 27B nên không vừa T4.
Bật `USE_DRIVE` để lưu model (SD + ControlNet + IP-Adapter ~6,3 GB, cộng Qwen) vào Google Drive, lần sau không tải lại.

⚠️ Hai link trycloudflare là **công khai, không có mật khẩu**: đừng chia sẻ; tắt runtime khi không dùng.

In [ ]:
USE_DRIVE = True  #@param {type:"boolean"}
ENABLE_LLM = True  #@param {type:"boolean"}
LLM_MODEL = "auto"  #@param ["auto", "qwen3.8:27b-q4_K_M", "qwen3.5:9b-q4_K_M", "qwen3.5:4b-q4_K_M"]

import os, subprocess
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_ROOT = '/content/drive/MyDrive/sketch_studio_models'
else:
    MODEL_ROOT = '/content/models_cache'
os.makedirs(MODEL_ROOT, exist_ok=True)

gpu, vram_mib = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits']).decode().strip().split(', ')
VRAM_GB = int(vram_mib) / 1024
if LLM_MODEL == 'auto':
    LLM_MODEL = 'qwen3.8:27b-q4_K_M' if VRAM_GB >= 38 else 'qwen3.5:9b-q4_K_M'
LLM_GB = {'qwen3.8:27b-q4_K_M': 18, 'qwen3.5:9b-q4_K_M': 6.6, 'qwen3.5:4b-q4_K_M': 3.3}.get(LLM_MODEL, 8) if ENABLE_LLM else 0
print(f'GPU: {gpu} ({VRAM_GB:.0f} GB) — LLM: {LLM_MODEL if ENABLE_LLM else "tắt"}')
if ENABLE_LLM and LLM_GB + 5 > VRAM_GB:
    print('⚠️ Không đủ VRAM cho cả LLM và ComfyUI: một phần model sẽ chạy trên CPU (chậm). Cân nhắc model nhỏ hơn hoặc GPU lớn hơn.')

In [ ]:
# Cài ComfyUI (Colab đã có sẵn torch + CUDA, requirements sẽ không cài lại torch)
%cd /content
!test -d ComfyUI || git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
# IP-Adapter (khoá gương mặt nhân vật cho biến thể & bộ biểu cảm). Repo ở chế độ "maintenance only" từ 4/2025:
# nếu ComfyUI mới làm hỏng node này, web/Claude tự bỏ khoá mặt thay vì lỗi.
!test -d custom_nodes/ComfyUI_IPAdapter_plus || git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus.git custom_nodes/ComfyUI_IPAdapter_plus

In [ ]:
# Tải model (bỏ qua file đã có trong cache/Drive) rồi link vào ComfyUI/models
MODELS = {
    'checkpoints/DreamShaper_8_pruned.safetensors':
        'https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors',
    'controlnet/control_v11p_sd15_lineart_fp16.safetensors':
        'https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_lineart_fp16.safetensors',
    'controlnet/control_v11p_sd15_scribble_fp16.safetensors':
        'https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_scribble_fp16.safetensors',
    # IP-Adapter: tên file phải đúng như dưới để Unified Loader tìm thấy
    'ipadapter/ip-adapter-plus-face_sd15.safetensors':
        'https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus-face_sd15.safetensors',
    'clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors':
        'https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors',
}
import subprocess
for rel, url in MODELS.items():
    cached = os.path.join(MODEL_ROOT, rel)
    os.makedirs(os.path.dirname(cached), exist_ok=True)
    if not os.path.exists(cached) or os.path.getsize(cached) < 50_000_000:
        print('Tải', rel)
        subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', cached, url], check=True)
    target = os.path.join('/content/ComfyUI/models', rel)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        os.symlink(cached, target)
    print('OK', rel, round(os.path.getsize(cached) / 1e9, 2), 'GB')

In [ ]:
# Khởi động ComfyUI (nền) + cloudflared tunnel, in ra COMFY_URL
import re, socket, subprocess, threading, time

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

def port_open(port):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

if not port_open(8188):
    comfy_log = open('/content/comfy.log', 'w')
    args = ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--preview-method', 'none']
    if ENABLE_LLM:  # leave VRAM for Ollama on the same GPU
        args += ['--reserve-vram', str(round(LLM_GB + 1.5, 1))]
    comfy = subprocess.Popen(args, cwd='/content/ComfyUI', stdout=comfy_log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if port_open(8188): break
        if comfy.poll() is not None: raise RuntimeError(open('/content/comfy.log').read()[-3000:])
        time.sleep(1)
print('ComfyUI chạy ở 127.0.0.1:8188')

def open_tunnel(port, host_header=None):
    cmd = ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--no-autoupdate']
    if host_header:
        cmd += ['--http-host-header', host_header]
    proc = subprocess.Popen(cmd, stderr=subprocess.PIPE, text=True)
    for line in proc.stderr:
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
        if m:
            threading.Thread(target=lambda: [None for _ in proc.stderr], daemon=True).start()  # keep draining logs
            return m.group(0)

COMFY_URL = open_tunnel(8188)

print('COMFY_URL =', COMFY_URL)


In [ ]:
# Ollama + Qwen (đọc ảnh, đọc chữ tiếng Việt, gợi ý) — bỏ qua nếu ENABLE_LLM = False
import json, urllib.request
LLM_URL = None
if ENABLE_LLM:
    os.environ['OLLAMA_MODELS'] = os.path.join(MODEL_ROOT, 'ollama')  # cache model trên Drive
    os.environ['OLLAMA_KEEP_ALIVE'] = '30m'
    os.environ['OLLAMA_CONTEXT_LENGTH'] = '16384'
    if not os.path.exists('/usr/local/bin/ollama'):
        !apt-get -qq install -y zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh
    if not port_open(11434):
        subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT, env=os.environ.copy())
        for _ in range(60):
            if port_open(11434): break
            time.sleep(1)
    !ollama pull {LLM_MODEL}
    print('Nạp model vào GPU (lần đầu 1–3 phút)…')
    req = urllib.request.Request('http://127.0.0.1:11434/api/generate', headers={'Content-Type': 'application/json'},
                                 data=json.dumps({'model': LLM_MODEL, 'prompt': 'Xin chào', 'stream': False, 'think': False}).encode())
    print(json.loads(urllib.request.urlopen(req, timeout=900).read())['response'][:80])
    LLM_URL = open_tunnel(11434, host_header='localhost:11434')  # Ollama chỉ nhận Host localhost

print('\n' + '=' * 70)
print('COMFY_URL =', COMFY_URL)
print('LLM_URL   =', LLM_URL, f'({LLM_MODEL})' if LLM_URL else '')
print('Dán 2 link vào web studio → ⚙︎ Cài đặt (hoặc nói với Claude Desktop: "đặt comfy url là ...")')
print('=' * 70)

In [ ]:
# (Tuỳ chọn) Xem log khi có lỗi
!tail -n 30 /content/comfy.log
!tail -n 15 /content/ollama.log 2>/dev/null